## Generating summary & bias for each article:

In [2]:
import sys
sys.path.append("/data/cb/scratch/bfefferm/NLP-Project/hfppl")

In [4]:
import pandas as pd
import csv
import os
from hfppl import Model, LMContext, TokenCategorical, CachedCausalLM, smc_steer, smc_standard
from score import compute_pbf_score, compute_pbi_score
from smc_steer_summary import bias_model_factory, TwistModel, gen_summary

**Loading Dataset:**

In [5]:
dataset = pd.read_csv('../POLITICS_finetuning/processed_data.csv')

In [6]:
dataset

,title,body,stance
0,Elizabeth Cheney blasted by older sister over ...,"Call it Cheney versus Cheney.\nMary Cheney, on...",center
1,Mary Cheney: Sister Is 'Dead Wrong' On Gay Mar...,"Mary Cheney, the younger sister of Wyoming U.S...",center
2,IRS official who refused to testify facing mor...,The IRS official who refused to testify at a H...,center
3,White House Plays Down Data Program,WASHINGTON — The Obama administration tried Sa...,center
4,N.R.A. Details Plan for Armed School Guards,Report Sees Guns as Path to Safety in Schools\...,center
...,...,...,...
295,Nancy Pelosi Re-Elected House Minority Leader,WASHINGTON ― House Minority Leader Nancy Pelos...,right
296,Nancy Pelosi Beats Back House Democratic Leade...,WASHINGTON — House Democrats on Wednesday reje...,center
297,Obama Will Meet With Sanders On Thursday,WASHINGTON -- With presumptive Democratic pres...,left
298,"Clinton Is 'Sane' And 'Competent,' Unlike Trum...",PHILADELPHIA ― Americans should vote for Hilla...,center


**Specifying file paths:**

In [7]:
# Model paths:
bias_model_path = '/data/cb/scratch/bfefferm/NLP-Project-storage/Saved_Models/politics_best_2500/politics_best_2500_30bz_000007_best'
bias_tokenizer_path = '/data/cb/scratch/bfefferm/NLP-Project-storage/Saved_Models/politics_best_2500/tokenizer_politics_best_2500_30bz_000007_best'

bias_model = bias_model_factory(bias_model_path, bias_tokenizer_path)

# Specifying model name:
llm_model_name = 'gpt2'

# Loading llm:
llm = CachedCausalLM.from_pretrained(llm_model_name)

2023-11-23 17:12:19.786776: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-11-23 17:12:19.786821: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-11-23 17:12:19.786880: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-11-23 17:12:19.797255: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-11-23 17:12:21.655675: W tensorflow/c

**Iterating over each summary, predicting bias, & saving to `.csv`:**

In [10]:
num_summaries = 3

In [12]:
# Open file  
with open('../POLITICS_finetuning/processed_data.csv') as file_obj: 
    # Create reader object by passing the file  
    # object to reader method 
    reader_obj = csv.reader(file_obj) 
    with open('../POLITICS_finetuning/summariesAndBiases.csv', 'w') as f:  
        # Initialize writer object:
        writer_obj = csv.writer(f)
        # The fields of this file are
        # ['title', 'body', 'stance']
        # Iterate over each row in the .csv,
        # skipping the first row (pertaining to field / column)
        next(reader_obj)
        for row in reader_obj: 
            # Store title:
            title = row[0]
            # Store article:
            article = row[1]
            # Ensuring that articles fit within maximum length
            # article = article[:llm.tokenizer.model_max_length] 
            # For each stance:
            writer_obj.writerow(['Title', 'Summary', 'Predicted Bias', 'Stance'])
            for stance in ['left', 'center', 'right']:
                for i in range(num_summaries):
                    summary = await gen_summary(llm_model_name, llm, bias_model, TwistModel, article, stance)
                    # For each summary, predict its bias:
                    pred_bias, logits = bias_model(summary)
                    writer_obj.writerow([title, summary, pred_bias, stance])
                    
    writer_obj.close()
    
reader.close()

CancelledError: 

Exception in callback CachedCausalLM.add_query.<locals>.<lambda>() at /data/cb/bfefferm/mambaforge/envs/jnb/lib/python3.11/site-packages/hfppl/llms.py:321
handle: <TimerHandle when=1554457.644558662 CachedCausalLM.add_query.<locals>.<lambda>() at /data/cb/bfefferm/mambaforge/envs/jnb/lib/python3.11/site-packages/hfppl/llms.py:321>
Traceback (most recent call last):
  File "/data/cb/bfefferm/mambaforge/envs/jnb/lib/python3.11/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "/data/cb/bfefferm/mambaforge/envs/jnb/lib/python3.11/site-packages/hfppl/llms.py", line 321, in <lambda>
    self.timer = asyncio.get_running_loop().call_later(self.timeout, lambda: self.batch_evaluate_queries())
                                                                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data/cb/bfefferm/mambaforge/envs/jnb/lib/python3.11/site-packages/torch/utils/_contextlib.py", line 115, in decorate_context
    return func(*args